# MTG-Causal-RL: Getting Started Tutorial

Welcome to the **MTG-Causal-RL** benchmark! This tutorial will guide you through:

1. **Environment Overview** - Game mechanics and observation/action spaces
2. **Available Archetypes** - The 5 Standard 2025 competitive decks
3. **The Structural Causal Model** - Our explicit SCM for MTG strategy
4. **Running Baseline Agents** - Random, Heuristic, and learned agents
5. **The Agent Registry** - Managing and accessing agents by name
6. **Creating Custom Agents** - Build, register, and run your own agents
7. **Command Line Usage** - Training and evaluation scripts

---

## Prerequisites

Make sure you have installed the package:

```bash
uv pip install -e ".[dev]"
```


In [ ]:
# Core imports
import matplotlib.pyplot as plt
import numpy as np

from mtg.agents import (
    BaseAgent,
    CausalAgent,
    RandomAgent,
    get_agent,
    list_agents,
    register_agent,
)
from mtg.agents.heuristics import GreedyAggroAgent
from mtg.causal.scm import StructuralCausalModel

# MTG-Causal-RL imports
from mtg.env import MTGEnv
from mtg.env.deck_archetypes import get_archetype, list_archetypes

print("All imports successful!")
print(f"Available agents: {list_agents()}")

## 1. Environment Overview

The MTG-Causal-RL environment simulates strategic Magic: The Gathering gameplay with a configurable turn horizon. Let's create an environment and explore its properties.


In [ ]:
# Create the environment
env = MTGEnv(
    deck_archetype="mono_red_aggro",
    opponent_archetype="azorius_control",
    max_turns=10,
    reward_type="shaped",
    render_mode="ansi",
    seed=42,
)

print(f"Observation Space: {env.observation_space}")
print(f"Action Space: {env.action_space}")
print(f"\nPlayer Deck: {env.deck_archetype_name}")
print(f"Opponent Deck: {env.opponent_archetype_name}")

In [ ]:
# Reset and view initial state
obs, info = env.reset(seed=42)

print("Initial Game State:")
print(f"  Turn: {info['turn']}")
print(f"  Phase: {info['phase']}")
print(f"  Life: {info['life']} vs Opponent: {info['opponent_life']}")
print(f"  Hand Size: {info['hand_size']}")
print(f"\nAction Mask Shape: {info['action_mask'].shape}")
print(f"Legal Actions: {np.where(info['action_mask'] > 0)[0]}")

## 2. Available Archetypes

The benchmark includes 5 Standard 2025 competitive archetypes.


In [ ]:
# List all archetypes
archetypes = list_archetypes()
print("Available Archetypes:")
print("=" * 50)

for name in archetypes:
    arch = get_archetype(name)
    print(f"\n{arch.display_name}")
    print(f"   Strategy: {arch.strategy.name}")
    print(f"   Tier: {arch.tier}")
    print(f"   Meta Share: {arch.meta_share:.0%}")
    print(f"   Colors: {', '.join(arch.colors)}")

## 3. The Structural Causal Model

The core contribution of this benchmark is the explicit causal model for MTG strategy.


In [ ]:
from mtg.causal.scm import CausalLayer

# Create the SCM
scm = StructuralCausalModel()

# Display the causal variables
print("Causal Variables in MTG-Causal-RL:")
print("=" * 60)

for layer in [
    CausalLayer.RESOURCE,
    CausalLayer.BOARD_STATE,
    CausalLayer.STRATEGIC,
    CausalLayer.OUTCOME,
]:
    vars_in_layer = scm.variables.list_by_layer(layer)
    print(f"\n{layer.name} LAYER")
    for var in vars_in_layer:
        parents = ", ".join(var.parents) if var.parents else "(root)"
        print(f"   {var.name}: {var.description}")
        print(f"      ↳ Parents: {parents}")

## 4. Running Agents

Let's run different agents and compare their performance.


In [ ]:
def run_episode(env, agent, render=False):
    """Run a single episode and return reward."""
    obs, info = env.reset()
    total_reward = 0
    done = False

    while not done:
        action_mask = info["action_mask"]
        action = agent.select_action(obs, action_mask, info)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        done = terminated or truncated

        if render:
            env.render()

    return total_reward, info.get("game_result", "unknown")


def evaluate_agent(env, agent, n_episodes=100):
    """Evaluate an agent over multiple episodes."""
    wins = 0
    total_reward = 0

    for _ in range(n_episodes):
        reward, result = run_episode(env, agent)
        total_reward += reward
        if result == "win":
            wins += 1

    return {
        "win_rate": wins / n_episodes,
        "avg_reward": total_reward / n_episodes,
    }


# Create agents
random_agent = RandomAgent(seed=42)
greedy_agent = GreedyAggroAgent(aggression=0.7, seed=42)

# Evaluate Random Agent
print("Evaluating Random Agent...")
random_results = evaluate_agent(env, random_agent, n_episodes=50)
print(f"  Win Rate: {random_results['win_rate']:.1%}")
print(f"  Avg Reward: {random_results['avg_reward']:.2f}")

# Evaluate Greedy Aggro Agent
print("\nEvaluating Greedy Aggro Agent...")
heuristic_results = evaluate_agent(env, greedy_agent, n_episodes=50)
print(f"  Win Rate: {heuristic_results['win_rate']:.1%}")
print(f"  Avg Reward: {heuristic_results['avg_reward']:.2f}")

## 5. The Agent Registry

The benchmark uses a registry pattern for agents, making it easy to list, create, and extend agents.


In [ ]:
# List all registered agents
print("Registered Agents:")
print("=" * 40)
for name in list_agents():
    print(f"  • {name}")

# Create an agent using the registry
agent = get_agent("greedy_aggro", aggression=0.8, seed=123)
print(f"\nCreated agent: {agent}")
print(f"Agent name: {agent.name}")

## 6. Creating a Custom Agent

You can create your own agents by subclassing `BaseAgent` and registering them with the benchmark.


In [ ]:
# Step 1: Define a custom agent by subclassing BaseAgent
class AggressiveAgent(BaseAgent):
    """Custom agent that prioritizes attacking and dealing damage.

    This agent always attacks when possible and prefers casting
    damage-dealing spells over other actions.
    """

    def __init__(self, seed=None):
        super().__init__(name="AggressiveAgent", deterministic=False)
        self.rng = np.random.default_rng(seed)

    def select_action(self, observation, action_mask, info=None):
        """Select the most aggressive legal action."""
        legal = np.where(action_mask > 0)[0]

        if len(legal) == 0:
            return 0

        # Priority: Attack (13) > Cast spells (8-12) > Play land (3-5) > Pass (0)
        # Check for attack action
        if 13 in legal:
            return 13

        # Check for spell casting
        spell_actions = [a for a in legal if 8 <= a <= 12]
        if spell_actions:
            return spell_actions[0]

        # Check for land plays
        land_actions = [a for a in legal if 3 <= a <= 5]
        if land_actions:
            return land_actions[0]

        # Keep hand in mulligan
        if 1 in legal:
            return 1

        # Default: random from remaining
        return int(self.rng.choice(legal))


print("AggressiveAgent class defined")

In [ ]:
# Step 2: Register the custom agent
register_agent("aggressive", AggressiveAgent)

# Verify it's registered
print("Updated agent list:")
for name in list_agents():
    marker = "⭐ NEW" if name == "aggressive" else ""
    print(f"  • {name} {marker}")

In [ ]:
# Step 3: Use the custom agent via the registry
my_agent = get_agent("aggressive", seed=42)

# Evaluate our custom agent
print("Evaluating AggressiveAgent...")
aggressive_results = evaluate_agent(env, my_agent, n_episodes=50)
print(f"  Win Rate: {aggressive_results['win_rate']:.1%}")
print(f"  Avg Reward: {aggressive_results['avg_reward']:.2f}")

# Compare with baselines
print("\n" + "=" * 50)
print("Comparison:")
print("=" * 50)
print(f"{'Agent':<20} {'Win Rate':<15} {'Avg Reward':<15}")
print("-" * 50)
wr = random_results["win_rate"]
ar = random_results["avg_reward"]
print(f"{'Random':<20} {wr:.1%}{'':<10} {ar:.2f}")
wr = heuristic_results["win_rate"]
ar = heuristic_results["avg_reward"]
print(f"{'Heuristic':<20} {wr:.1%}{'':<10} {ar:.2f}")
wr = aggressive_results["win_rate"]
ar = aggressive_results["avg_reward"]
print(f"{'Aggressive (ours)':<20} {wr:.1%}{'':<10} {ar:.2f}")

## 7. The Causal Agent with Decision Logging

The `CausalAgent` combines a PPO policy with SCM-based counterfactual reasoning. It can log decisions for paper analysis.

> **Note:** The paper's headline contribution is the closely related but distinct `CGFAAgent` (CGFA-PPO), which factors PPO's value head and advantage estimator along the SCM and adds a learnable residual gate plus an intervention-calibration loss. See the [CGFA section of the README](../README.md#causal-graph-factored-advantage-cgfa-ppo) and the [CGFA paper pipeline](../README.md#5-the-cgfa-paper-pipeline) for the full story; this tutorial cell focuses on the simpler `CausalAgent` baseline.


In [ ]:
# Create a CausalAgent with decision logging
# CausalAgent was already imported in cell 1

# Get observation/action dimensions
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n

# Create agent with logging enabled
causal_agent = CausalAgent(
    observation_dim=obs_dim,
    action_dim=act_dim,
    causal_weight=0.6,  # 0=pure RL, 1=pure causal
    log_decisions=True,  # Enable decision logging
    seed=42,
)

# Initialize the base model
causal_agent.initialize_model(env)

# Run a few episodes
print("Running CausalAgent with decision logging...")
obs, info = env.reset()
for _ in range(20):
    action = causal_agent.select_action(obs, info["action_mask"], info)
    obs, _, done, _, info = env.step(action)
    if done:
        obs, info = env.reset()

# Get causal reasoning statistics
stats = causal_agent.get_causal_stats()
print("\nCausal Decision Statistics:")
print(f"  Total decisions: {stats.get('total_decisions', 0)}")
print(f"  Avg causal effect: {stats.get('avg_causal_effect', 0):.4f}")
print(f"  Causal preferred ratio: {stats.get('causal_preferred_ratio', 0):.1%}")

# Save decision log for paper analysis
# causal_agent.save_decision_log("results/causal_decisions.json")

## 8. Publication-Grade Visualizations

MTG-Causal-RL includes beautiful visualizations for papers and presentations.


In [ ]:
# Import visualization utilities
from mtg.utils.visualization import (
    apply_publication_style,
    create_comparison_bar,
    create_learning_curve,
    create_scm_diagram,
)

# Apply publication style
apply_publication_style()

# Generate the SCM diagram
print("Creating SCM Diagram...")
fig = create_scm_diagram()
plt.show()
print("This diagram is publication-ready at 300 DPI!")

In [ ]:
# Create a sample learning curve comparison
print("Creating Learning Curves...")

# Sample data (you would use real training data)
steps = np.arange(0, 100001, 5000)

data = {
    "Random": {
        "steps": steps,
        "mean": np.ones_like(steps, dtype=float) * 0.25,
        "std": np.ones_like(steps, dtype=float) * 0.03,
    },
    "Heuristic": {
        "steps": steps,
        "mean": np.ones_like(steps, dtype=float) * 0.42,
        "std": np.ones_like(steps, dtype=float) * 0.02,
    },
    "PPO": {
        "steps": steps,
        "mean": 0.25 + 0.35 * (1 - np.exp(-steps / 30000)),
        "std": 0.04 * np.exp(-steps / 50000) + 0.02,
    },
    "Causal": {
        "steps": steps,
        "mean": 0.25 + 0.45 * (1 - np.exp(-steps / 20000)),
        "std": 0.03 * np.exp(-steps / 40000) + 0.015,
    },
}

fig = create_learning_curve(
    data=data,
    metric="win_rate",
    title="Win Rate vs Training Steps",
)
plt.show()

print("Learning curves with confidence bands and styled markers!")

In [ ]:
# Create agent comparison bar chart
print("Creating Comparison Bar Chart...")

results = {
    "Random": {"win_rate": 0.251, "win_rate_std": 0.028, "avg_reward": -0.498},
    "Heuristic": {"win_rate": 0.423, "win_rate_std": 0.031, "avg_reward": -0.154},
    "PPO": {"win_rate": 0.587, "win_rate_std": 0.024, "avg_reward": 0.174},
    "Causal": {"win_rate": 0.672, "win_rate_std": 0.019, "avg_reward": 0.344},
}

fig = create_comparison_bar(
    results=results,
    metrics=["win_rate"],
    title="Agent Performance Comparison",
)
plt.show()

print("Bar charts with error bars and value labels!")

## 9. Command-Line Workflows

MTG-Causal-RL provides three day-to-day CLI workflows (training, evaluation,
gameplay) plus a composable **research pipeline** for multi-seed sweeps and
paired-bootstrap statistical comparisons. Each workflow has **interactive**
and **command-line** modes; the research pipeline is non-interactive by design.

### Training Workflow
```bash
# Interactive mode (guided prompts)
uv run python scripts/runner/run_training.py --interactive

# Vanilla PPO baseline (1M steps)
uv run python scripts/runner/run_training.py \
    --agent ppo --timesteps 1000000 \
    --deck mono_red_aggro --opponent azorius_control

# Multi-opponent round-robin training (default mode)
uv run python scripts/runner/run_training.py \
    --agent ppo --deck mono_red_aggro --opponent all --timesteps 2000000

# Causal RL agent (same setup, adds learned causal world model)
uv run python scripts/runner/run_training.py \
    --agent causal --deck mono_red_aggro --opponent all --timesteps 2000000

# Causal RL with curriculum training (auto 70% → full agency 30%)
# Curriculum is designed for the Causal RL agent, which leverages learned
# causal effects to handle the expanded action space in Phase 2.
uv run python scripts/runner/run_training.py \
    --agent causal --agency curriculum --timesteps 2000000

# Sequential training for ablation (full budget per opponent)
uv run python scripts/runner/run_training.py \
    --opponent all --training-mode sequential --timesteps 500000
```

### Evaluation Workflow
```bash
# Interactive mode (guided prompts)
uv run python scripts/runner/run_evaluation.py --interactive

# Evaluate all agents (episodes are per opponent, split only across seeds)
uv run python scripts/runner/run_evaluation.py --agent all --episodes 500

# Evaluate with sample report generation
uv run python scripts/runner/run_evaluation.py --agent ppo --show-games 1 --save-reports
```

### Gameplay Workflow
```bash
# Interactive mode (full configuration)
uv run python scripts/runner/run_gameplay.py --interactive

# Quick demo game
uv run python scripts/runner/run_gameplay.py --demo

# Custom matchup
uv run python scripts/runner/run_gameplay.py \
    --player-agent heuristic --player-deck mono_red_aggro \
    --opponent-agent heuristic --opponent-deck dimir_midrange
```

### Plot Regeneration
```bash
# Re-render training_curves.png + evaluation_results.png from a saved run
uv run python -m scripts.runner.regenerate_plots \
    results/trained_agents/<run_name>
```

### Research Pipeline (`mtg-research`, recommended for publication)

For multi-seed sweeps and paired-bootstrap statistical comparisons, use
`mtg-research`: a single CLI that unifies the three-stage pipeline with
subcommands and an interactive wizard. Full guide:
[Research Pipeline (Section 4 of the main README)](../README.md#4-research-pipeline-mtg-research).

**The easiest entry point is the interactive wizard:**

```bash
# Walks you through every setting, then runs train + eval + aggregate.
# Pick option 2 for a paper-quality PPO vs Causal paired comparison.
uv run mtg-research -i
```

**Scripted equivalent** (end-to-end PPO vs Causal A/B comparison):

```bash
# PPO baseline end-to-end at 3 seeds. Each player deck is auto-paired
# with `random` + its canonical heuristic (e.g. mono_red_aggro -> random
# + greedy_aggro; azorius_control -> random + control). No need to pick
# baselines manually.
uv run mtg-research pipeline \
    --experiment-name ppo_baseline_v1 --agents ppo \
    --seeds 42 123 456 --timesteps-per-opponent 2000000 --agency auto \
    --eval-episodes 500

# Causal RL end-to-end at the SAME seeds (paired comparison).
# `--no-baselines` here just avoids re-evaluating identical baselines.
uv run mtg-research pipeline \
    --experiment-name causal_v1 --agents causal \
    --seeds 42 123 456 --timesteps-per-opponent 2000000 --agency curriculum \
    --eval-episodes 500 --no-baselines

# Cross-sweep paired-bootstrap significance tests
uv run mtg-research aggregate \
    --eval-results results/research/ppo_baseline_v1/eval/eval_results.json \
                   results/research/causal_v1/eval/eval_results.json \
    --source-labels ppo causal --baseline-agent ppo \
    --output-dir results/research/comparison_ppo_vs_causal
```

Each stage is **resumable** (re-running skips completed work) and writes
machine-readable artifacts so any later stage can be re-run independently.
Stages can also be run individually: `mtg-research train`,
`mtg-research eval`, `mtg-research aggregate`.

**Smoke-test the whole pipeline in ~3 minutes:**

```bash
# Interactive: pick option 3 (Quick smoke test)
uv run mtg-research -i

# Or scripted
uv run mtg-research train --quick
uv run mtg-research eval results/research/smoke_test \
    --eval-episodes 10 --baseline-agents random   # tiny override for speed
uv run mtg-research aggregate \
    --eval-results results/research/smoke_test/eval/eval_results.json \
    --output-dir results/research/smoke_test/aggregated --baseline-agent ppo
```

### CGFA Paper Pipeline (`mtg-research {ablation,transfer,calibration-plot,case-study}`)

These four subcommands produce the artefacts used in the CGFA-PPO
publication. Full guide:
[Section 5 of the main README](../README.md#5-the-cgfa-paper-pipeline).

```bash
# 1. Six-point ablation suite (PPO vs Causal vs CGFA-scalar-only vs
#    CGFA-no-gate vs CGFA-no-cal vs CGFA-full) at matched seeds.
uv run mtg-research ablation \
    --experiment-name cgfa_ablation_v1 \
    --player-decks mono_red_aggro \
    --seeds 42 123 456 \
    --opponents mono_red_aggro azorius_control dimir_midrange \
    --timesteps-per-opponent 1000000 \
    --eval-episodes 500

# 2. Transfer experiment: train on K opponents, evaluate on a disjoint
#    held-out set; report a per-agent generalisation gap with a
#    paired-bootstrap CI / p-value.
uv run mtg-research transfer \
    --experiment-name cgfa_transfer_v1 \
    --agents ppo cgfa --player-decks mono_red_aggro --seeds 42 123 456 \
    --train-opponents mono_red_aggro azorius_control dimir_midrange \
    --heldout-opponents domain_ramp boros_convoke \
    --timesteps-per-opponent 1000000 --eval-episodes 500

# 3. Calibration plot: render Pearson(A_k, eps_k), per-factor credit
#    share, and gate trajectory from a CGFA training log.
uv run mtg-research calibration-plot \
    results/research/cgfa_v1/<run>/cgfa/cgfa_calibration.csv \
    --output results/figures/cgfa_calibration.png

# 4. Case study: deterministic per-step factor attribution from a
#    saved CGFA model.
uv run mtg-research case-study \
    --model results/trained_agents/<cgfa_run>/best_model.zip \
    --player-deck mono_red_aggro --opponent-deck azorius_control \
    --episode-seed 7 \
    --csv results/case_study/episode7.csv \
    --figure results/case_study/episode7.png
```

### SCM Diagram (programmatic)

The Structural Causal Model diagram is generated directly from
`mtg.utils.visualization`:

```python
from mtg.utils.visualization import create_scm_diagram

fig = create_scm_diagram()
fig.savefig("scm_diagram.pdf", dpi=300, bbox_inches="tight")
```

All experiment figures (win-rate bars, headline comparison, per-matchup
heatmap, LaTeX tables, paired-bootstrap significance) are produced by the
Research Pipeline above.

The CLI includes:
- Animated ASCII logo
- Live training progress with sparklines
- Phase-by-phase game visualization
- Beautiful result tables with statistics


## 10. HTML Gameplay Reports

MTG-Causal-RL can generate interactive HTML reports of game sessions for post-hoc analysis:

```python
from mtg.utils import GameRecorder, generate_html_report
from pathlib import Path

# Create a recorder
recorder = GameRecorder(
    player_deck="Mono-Red Aggro",
    opponent_deck="Azorius Control",
    player_agent="PPO",
    opponent_agent="Heuristic",
)

# Record game events
recorder.set_player_on_play(True)
recorder.record_action(1, "Main 1", "Player", "PLAY_LAND", "Play Mountain")
recorder.record_action(1, "Main 1", "Player", "CAST", "Cast Lightning Bolt")

# Record state snapshots (with creatures and graveyard)
recorder.record_snapshot(
    turn=1, phase="End", active_player="Player",
    player_life=20, opponent_life=17,
    player_hand_size=6, opponent_hand_size=7,
    player_lands=1, opponent_lands=0,
    player_creatures=[("Swiftspear", 1, 2, False)],
    opponent_creatures=[],
    player_graveyard=[("Lightning Bolt", "instant")],
    opponent_graveyard=[],
)

# Generate the HTML report
replay = recorder.get_replay()
generate_html_report(replay, Path("results/gameplay/my_game.html"))
```

The CLI displays:
- **Creatures** with power/toughness and tapped status
- **Graveyard** with card type icons and counts
- **Phase indicators** showing turn progression

To generate reports from evaluation:

```bash
# Evaluate with HTML reports
uv run python scripts/runner/run_evaluation.py --show-games 3 --save-reports

# View generated evaluation reports
open results/evaluations/*/reports/*/replay.html
```


---

## Summary & Next Steps

You've learned how to:
- Create and interact with the MTG environment
- Use the 5 Standard 2025 competitive archetypes
- Explore the Structural Causal Model
- Run and evaluate baseline agents
- Create and register custom agents
- Generate polished figures suitable for papers
- Use the beautiful CLI interface
- Generate HTML gameplay reports

### Next Steps

1. **Extend with Learning**: Add a `learn()` method to your agent for online learning
2. **Add Custom Cards**: Use `CardRegistry` to add new cards
3. **Create New Archetypes**: Use `register_custom_archetype()` for new decks
4. **Extend the SCM**: Add custom causal variables
5. **Run a Research Sweep**: `uv run mtg-research -i` for the interactive wizard, or `mtg-research pipeline ...` for scripted runs (see [Research Pipeline in the main README](../README.md#4-research-pipeline-mtg-research))

### Quick Commands

```bash
# Full demo with HTML reports
uv run python scripts/runner/run_gameplay.py --interactive

# Train a PPO agent
uv run python scripts/runner/run_training.py --agent ppo --timesteps 1000000

# Evaluate with sample reports
uv run python scripts/runner/run_evaluation.py --agent all --show-games 1 --save-reports

# Multi-seed research sweep, interactive (easiest)
uv run mtg-research -i

# Multi-seed research sweep, scripted (PPO end-to-end at 3 seeds)
uv run mtg-research pipeline \
    --experiment-name ppo_baseline_v1 --agents ppo \
    --seeds 42 123 456 --timesteps-per-opponent 2000000 --agency auto \
    --eval-episodes 500   # auto-pairs random + canonical heuristic per deck
```

---

**For more details, see the [README](../README.md) and [API documentation](https://github.com/anonymous/mtg-causal-rl)**

Happy researching with MTG-Causal-RL!
